# 21 · Long-running fulfillment jobs

LongRunningFunctionTool returns an intermediate result. Your application later supplies the completion using the original function-call ID; ADK does not create a job queue for you.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Lite model calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.tools import LongRunningFunctionTool

def start_invoice_batch(customer_id: str) -> dict:
    """Start a simulated invoice export job and immediately return its pending status."""
    return {"job_id": "demo-job-001", "status": "pending", "customer_id": customer_id}

lab = await make_lab(llm("batch_assistant", "Start the batch once. If pending, tell the user it is pending and do not call again.",
                        tools=[LongRunningFunctionTool(start_invoice_batch)]))
events = await ask(lab, "Start the invoice batch for C101.")
calls = [c for e in events for c in e.get_function_calls() if c.name == "start_invoice_batch"]
assert calls
call = calls[0]
print("Correlation ID:", call.id)


In [ ]:
# Simulate a worker completion notification; no waiting or external queue.
await ask(lab, parts=[types.Part(function_response=types.FunctionResponse(
    id=call.id, name=call.name,
    response={"job_id": "demo-job-001", "status": "completed", "invoice_ids": ["INV1001"]}))])


## Try it

Return a failed job status and ensure the agent explains the failure without restarting the job.


## Reference

[Official ADK documentation](https://adk.dev/). Shared configuration: `config.json`. No environment activation or login cells are needed.
